In [1]:
import pandas as pd
from pathlib import Path

# Project paths
BASE_DIR = Path.cwd().parent

OUTPUT_DIR = BASE_DIR / "data" / "synthetic" / "seasonality_adjusted"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Approved seasonal multipliers
seasonality = pd.DataFrame({
    "month": range(1, 13),
    "month_name": [
        "January",
        "February",
        "March",
        "April",
        "May",
        "June",
        "July",
        "August",
        "September",
        "October",
        "November",
        "December"
    ],
    "seasonal_multiplier": [
        0.95,
        0.90,
        1.00,
        1.02,
        1.05,
        1.08,
        1.03,
        1.08,
        1.15,
        1.35,
        1.45,
        1.60
    ]
})

# Classify season
seasonality["season_type"] = [
    "Post-Holiday",
    "Soft Baseline",
    "Normal",
    "Modest Uplift",
    "Summer Uplift",
    "Mid-Year Activity",
    "Moderation",
    "Festival Build-Up",
    "Pre-Festival",
    "Festive Peak",
    "Festive Peak",
    "Year-End Peak"
]

print("02.12 Seasonality Control Plan")
print("=" * 50)

print(seasonality.to_string(index=False))

# Validation
print("\nValidation")
print("=" * 50)

print("Months:", len(seasonality))
print("Duplicate months:", seasonality["month"].duplicated().sum())
print("Missing multipliers:", seasonality["seasonal_multiplier"].isna().sum())
print(
    "Invalid multipliers:",
    (seasonality["seasonal_multiplier"] <= 0).sum()
)

# Save control plan
output_file = OUTPUT_DIR / "seasonality_control_plan.csv"
seasonality.to_csv(output_file, index=False)

print("\nControl plan saved to:")
print(output_file)

if (
    len(seasonality) == 12
    and seasonality["month"].nunique() == 12
    and seasonality["seasonal_multiplier"].notna().all()
    and (seasonality["seasonal_multiplier"] > 0).all()
):
    print("\n02.12 Step 1 validation: PASSED")
else:
    print("\n02.12 Step 1 validation: FAILED")

02.12 Seasonality Control Plan
 month month_name  seasonal_multiplier       season_type
     1    January                 0.95      Post-Holiday
     2   February                 0.90     Soft Baseline
     3      March                 1.00            Normal
     4      April                 1.02     Modest Uplift
     5        May                 1.05     Summer Uplift
     6       June                 1.08 Mid-Year Activity
     7       July                 1.03        Moderation
     8     August                 1.08 Festival Build-Up
     9  September                 1.15      Pre-Festival
    10    October                 1.35      Festive Peak
    11   November                 1.45      Festive Peak
    12   December                 1.60     Year-End Peak

Validation
Months: 12
Duplicate months: 0
Missing multipliers: 0
Invalid multipliers: 0

Control plan saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\seasonality_adjusted\seasonality_control_plan.cs

In [2]:
import pandas as pd
from pathlib import Path

# Project paths
BASE_DIR = Path.cwd().parent

SALES_DIR = BASE_DIR / "data" / "synthetic"
CONTROL_PLAN_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "seasonality_adjusted"
    / "seasonality_control_plan.csv"
)

# Load seasonality control plan
seasonality = pd.read_csv(CONTROL_PLAN_FILE)

# Load all monthly sales files
sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(sales_list, ignore_index=True)

print("Sales files loaded:", len(sales_files))
print("Total sales rows:", len(sales))

# Convert order date
sales["order_date"] = pd.to_datetime(sales["order_date"])

# Extract month
sales["month"] = sales["order_date"].dt.month

# Map seasonal multiplier
sales = sales.merge(
    seasonality[["month", "month_name", "seasonal_multiplier", "season_type"]],
    on="month",
    how="left"
)

# Calculate gross sales
sales["gross_sales_value"] = (
    sales["quantity"] * sales["unit_selling_price"]
)

# Monthly summary
monthly_seasonality = (
    sales
    .groupby(
        ["month", "month_name", "season_type", "seasonal_multiplier"],
        as_index=False
    )
    .agg(
        sales_lines=("sales_line_id", "count"),
        gross_sales=("gross_sales_value", "sum")
    )
    .sort_values("month")
)

# Calculate monthly sales contribution
monthly_seasonality["sales_contribution_pct"] = (
    monthly_seasonality["gross_sales"]
    / monthly_seasonality["gross_sales"].sum()
    * 100
)

print("\nMonthly Seasonality Profile")
print("=" * 80)

print(
    monthly_seasonality.to_string(index=False)
)

# Validation
print("\nValidation")
print("=" * 80)

print("Missing seasonal multipliers:",
      sales["seasonal_multiplier"].isna().sum())

print("Missing season types:",
      sales["season_type"].isna().sum())

print("Months represented:",
      sales["month"].nunique())

print("Total rows after mapping:",
      len(sales))

if (
    sales["seasonal_multiplier"].notna().all()
    and sales["season_type"].notna().all()
    and sales["month"].nunique() == 12
):
    print("\n02.12 Step 2 validation: PASSED")
else:
    print("\n02.12 Step 2 validation: FAILED")

Sales files loaded: 32
Total sales rows: 2118200

Monthly Seasonality Profile
 month month_name       season_type  seasonal_multiplier  sales_lines  gross_sales  sales_contribution_pct
     1    January      Post-Holiday                 0.95       172400 8.460534e+09                8.111609
     2   February     Soft Baseline                 0.90       163300 8.009442e+09                7.679121
     3      March            Normal                 1.00       181500 8.957821e+09                8.588388
     4      April     Modest Uplift                 1.02       185100 9.150704e+09                8.773316
     5        May     Summer Uplift                 1.05       190500 9.375555e+09                8.988893
     6       June Mid-Year Activity                 1.08       196000 9.663286e+09                9.264758
     7       July        Moderation                 1.03       186900 9.178243e+09                8.799719
     8     August Festival Build-Up                 1.08       196

In [3]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

# Approved monthly sales-line control totals
expected_volumes = {
    "2024-01": 53200,
    "2024-02": 50400,
    "2024-03": 56000,
    "2024-04": 57100,
    "2024-05": 58800,
    "2024-06": 60500,
    "2024-07": 57700,
    "2024-08": 60500,
    "2024-09": 64400,
    "2024-10": 75600,
    "2024-11": 81200,
    "2024-12": 89600,

    "2025-01": 57500,
    "2025-02": 54400,
    "2025-03": 60500,
    "2025-04": 61700,
    "2025-05": 63500,
    "2025-06": 65300,
    "2025-07": 62300,
    "2025-08": 65300,
    "2025-09": 69600,
    "2025-10": 81600,
    "2025-11": 87700,
    "2025-12": 96800,

    "2026-01": 61700,
    "2026-02": 58500,
    "2026-03": 65000,
    "2026-04": 66300,
    "2026-05": 68200,
    "2026-06": 70200,
    "2026-07": 66900,
    "2026-08": 70200
}

# Read actual monthly files
actual_volumes = {}

for file in sorted(SALES_DIR.glob("sales_*.csv")):
    month_key = file.stem.replace("sales_", "")
    actual_volumes[month_key] = len(pd.read_csv(file))

# Compare actual vs expected
validation = []

for month, expected in expected_volumes.items():
    actual = actual_volumes.get(month, 0)

    validation.append({
        "month": month,
        "expected_rows": expected,
        "actual_rows": actual,
        "difference": actual - expected,
        "status": "PASS" if actual == expected else "FAIL"
    })

volume_check = pd.DataFrame(validation)

print("02.12 STEP 3 — MONTHLY SEASONAL VOLUME VALIDATION")
print("=" * 70)

print(volume_check.to_string(index=False))

print("\nTotal expected rows:", volume_check["expected_rows"].sum())
print("Total actual rows:", volume_check["actual_rows"].sum())

failed = (volume_check["status"] == "FAIL").sum()

print("\nFailed months:", failed)

if failed == 0:
    print("\n02.12 Step 3 validation: PASSED")
else:
    print("\n02.12 Step 3 validation: FAILED")

02.12 STEP 3 — MONTHLY SEASONAL VOLUME VALIDATION
  month  expected_rows  actual_rows  difference status
2024-01          53200        53200           0   PASS
2024-02          50400        50400           0   PASS
2024-03          56000        56000           0   PASS
2024-04          57100        57100           0   PASS
2024-05          58800        58800           0   PASS
2024-06          60500        60500           0   PASS
2024-07          57700        57700           0   PASS
2024-08          60500        60500           0   PASS
2024-09          64400        64400           0   PASS
2024-10          75600        75600           0   PASS
2024-11          81200        81200           0   PASS
2024-12          89600        89600           0   PASS
2025-01          57500        57500           0   PASS
2025-02          54400        54400           0   PASS
2025-03          60500        60500           0   PASS
2025-04          61700        61700           0   PASS
2025-05        

In [4]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

monthly_data = []

for file in sorted(SALES_DIR.glob("sales_*.csv")):
    month_key = file.stem.replace("sales_", "")

    df = pd.read_csv(file)

    monthly_data.append({
        "month": month_key,
        "sales_lines": len(df)
    })

monthly_volume = pd.DataFrame(monthly_data)

# Extract month number
monthly_volume["month_number"] = (
    monthly_volume["month"]
    .str[-2:]
    .astype(int)
)

# Classify period
monthly_volume["period"] = monthly_volume["month_number"].apply(
    lambda x: "Festive_Year_End" if x in [10, 11, 12] else "Normal"
)

# Average monthly volume by period
period_summary = (
    monthly_volume
    .groupby("period", as_index=False)
    .agg(
        months=("month", "count"),
        average_monthly_sales_lines=("sales_lines", "mean"),
        total_sales_lines=("sales_lines", "sum")
    )
)

print("02.12 STEP 4 — FESTIVE & YEAR-END VALIDATION")
print("=" * 70)

print("\nMonthly volumes:")
print(
    monthly_volume[
        ["month", "sales_lines", "period"]
    ].to_string(index=False)
)

print("\nPeriod summary:")
print(period_summary.to_string(index=False))

# Extract averages
normal_avg = period_summary.loc[
    period_summary["period"] == "Normal",
    "average_monthly_sales_lines"
].iloc[0]

festive_avg = period_summary.loc[
    period_summary["period"] == "Festive_Year_End",
    "average_monthly_sales_lines"
].iloc[0]

increase_pct = (
    (festive_avg - normal_avg)
    / normal_avg
    * 100
)

print(f"\nNormal average monthly volume: {normal_avg:,.0f}")
print(f"Festive/year-end average volume: {festive_avg:,.0f}")
print(f"Festive/year-end increase: {increase_pct:.2f}%")

if festive_avg > normal_avg:
    print("\n02.12 Step 4 validation: PASSED")
else:
    print("\n02.12 Step 4 validation: FAILED")

02.12 STEP 4 — FESTIVE & YEAR-END VALIDATION

Monthly volumes:
  month  sales_lines           period
2024-01        53200           Normal
2024-02        50400           Normal
2024-03        56000           Normal
2024-04        57100           Normal
2024-05        58800           Normal
2024-06        60500           Normal
2024-07        57700           Normal
2024-08        60500           Normal
2024-09        64400           Normal
2024-10        75600 Festive_Year_End
2024-11        81200 Festive_Year_End
2024-12        89600 Festive_Year_End
2025-01        57500           Normal
2025-02        54400           Normal
2025-03        60500           Normal
2025-04        61700           Normal
2025-05        63500           Normal
2025-06        65300           Normal
2025-07        62300           Normal
2025-08        65300           Normal
2025-09        69600           Normal
2025-10        81600 Festive_Year_End
2025-11        87700 Festive_Year_End
2025-12        96800 Fest

In [5]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

monthly_data = []

for file in sorted(SALES_DIR.glob("sales_*.csv")):
    month_key = file.stem.replace("sales_", "")

    df = pd.read_csv(file)

    monthly_data.append({
        "month": month_key,
        "sales_lines": len(df)
    })

monthly_volume = pd.DataFrame(monthly_data)

# Extract month number
monthly_volume["month_number"] = (
    monthly_volume["month"]
    .str[-2:]
    .astype(int)
)

# Classify period
monthly_volume["period"] = monthly_volume["month_number"].apply(
    lambda x: "Festive_Year_End" if x in [10, 11, 12] else "Normal"
)

# Average monthly volume by period
period_summary = (
    monthly_volume
    .groupby("period", as_index=False)
    .agg(
        months=("month", "count"),
        average_monthly_sales_lines=("sales_lines", "mean"),
        total_sales_lines=("sales_lines", "sum")
    )
)

print("02.12 STEP 4 — FESTIVE & YEAR-END VALIDATION")
print("=" * 70)

print("\nMonthly volumes:")
print(
    monthly_volume[
        ["month", "sales_lines", "period"]
    ].to_string(index=False)
)

print("\nPeriod summary:")
print(period_summary.to_string(index=False))

# Extract averages
normal_avg = period_summary.loc[
    period_summary["period"] == "Normal",
    "average_monthly_sales_lines"
].iloc[0]

festive_avg = period_summary.loc[
    period_summary["period"] == "Festive_Year_End",
    "average_monthly_sales_lines"
].iloc[0]

increase_pct = (
    (festive_avg - normal_avg)
    / normal_avg
    * 100
)

print(f"\nNormal average monthly volume: {normal_avg:,.0f}")
print(f"Festive/year-end average volume: {festive_avg:,.0f}")
print(f"Festive/year-end increase: {increase_pct:.2f}%")

if festive_avg > normal_avg:
    print("\n02.12 Step 4 validation: PASSED")
else:
    print("\n02.12 Step 4 validation: FAILED")

02.12 STEP 4 — FESTIVE & YEAR-END VALIDATION

Monthly volumes:
  month  sales_lines           period
2024-01        53200           Normal
2024-02        50400           Normal
2024-03        56000           Normal
2024-04        57100           Normal
2024-05        58800           Normal
2024-06        60500           Normal
2024-07        57700           Normal
2024-08        60500           Normal
2024-09        64400           Normal
2024-10        75600 Festive_Year_End
2024-11        81200 Festive_Year_End
2024-12        89600 Festive_Year_End
2025-01        57500           Normal
2025-02        54400           Normal
2025-03        60500           Normal
2025-04        61700           Normal
2025-05        63500           Normal
2025-06        65300           Normal
2025-07        62300           Normal
2025-08        65300           Normal
2025-09        69600           Normal
2025-10        81600 Festive_Year_End
2025-11        87700 Festive_Year_End
2025-12        96800 Fest

In [6]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

SALES_DIR = BASE_DIR / "data" / "synthetic"
CONTROL_PLAN_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "seasonality_adjusted"
    / "seasonality_control_plan.csv"
)

# Load approved seasonality plan
seasonality = pd.read_csv(CONTROL_PLAN_FILE)

# Load monthly sales volumes
monthly_data = []

for file in sorted(SALES_DIR.glob("sales_*.csv")):
    month_key = file.stem.replace("sales_", "")

    df = pd.read_csv(file)

    monthly_data.append({
        "month": month_key,
        "month_number": int(month_key[-2:]),
        "sales_lines": len(df)
    })

monthly_volume = pd.DataFrame(monthly_data)

# Merge with approved multiplier
monthly_check = monthly_volume.merge(
    seasonality[["month", "month_name", "seasonal_multiplier"]],
    left_on="month_number",
    right_on="month",
    how="left"
)

monthly_check = monthly_check[
    ["month_x", "month_name", "seasonal_multiplier", "sales_lines"]
].rename(
    columns={"month_x": "period"}
)

print("02.12 STEP 5 — SEASONAL PATTERN VALIDATION")
print("=" * 70)

print(
    monthly_check.to_string(index=False)
)

# Year-wise correlation between multiplier and monthly volume
monthly_check["year"] = (
    monthly_check["period"]
    .str[:4]
    .astype(int)
)

correlations = (
    monthly_check
    .groupby("year")
    .apply(
        lambda x: x["seasonal_multiplier"].corr(
            x["sales_lines"]
        )
    )
    .reset_index(name="correlation")
)

print("\nCorrelation between approved multiplier and sales volume:")
print(correlations.to_string(index=False))

# Check expected direction:
# Higher seasonal multiplier should generally correspond
# to higher sales volume.

positive_years = (
    correlations["correlation"] > 0
).sum()

print("\nYears with positive seasonal relationship:", positive_years)

if positive_years == len(correlations):
    print("\n02.12 Step 5 validation: PASSED")
else:
    print("\n02.12 Step 5 validation: REVIEW")

02.12 STEP 5 — SEASONAL PATTERN VALIDATION
 period month_name  seasonal_multiplier  sales_lines
2024-01    January                 0.95        53200
2024-02   February                 0.90        50400
2024-03      March                 1.00        56000
2024-04      April                 1.02        57100
2024-05        May                 1.05        58800
2024-06       June                 1.08        60500
2024-07       July                 1.03        57700
2024-08     August                 1.08        60500
2024-09  September                 1.15        64400
2024-10    October                 1.35        75600
2024-11   November                 1.45        81200
2024-12   December                 1.60        89600
2025-01    January                 0.95        57500
2025-02   February                 0.90        54400
2025-03      March                 1.00        60500
2025-04      April                 1.02        61700
2025-05        May                 1.05        63500
202

In [7]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

CONTROL_PLAN_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "seasonality_adjusted"
    / "seasonality_control_plan.csv"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "seasonality_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load seasonality control plan
seasonality = pd.read_csv(CONTROL_PLAN_FILE)

# Required columns
required_columns = [
    "month",
    "month_name",
    "seasonal_multiplier",
    "season_type"
]

# Validate columns
missing_columns = [
    col for col in required_columns
    if col not in seasonality.columns
]

# Validate values
duplicate_months = seasonality["month"].duplicated().sum()
missing_values = seasonality[required_columns].isna().sum().sum()
invalid_multipliers = (
    seasonality["seasonal_multiplier"] <= 0
).sum()

print("02.12 STEP 6 — FINAL SEASONALITY PLAN VALIDATION")
print("=" * 70)

print("Rows:", len(seasonality))
print("Missing required columns:", len(missing_columns))
print("Duplicate months:", duplicate_months)
print("Missing values:", missing_values)
print("Invalid multipliers:", invalid_multipliers)

print("\nFinal Seasonality Plan:")
print(
    seasonality[required_columns]
    .to_string(index=False)
)

# Save final validated plan
FINAL_FILE = (
    OUTPUT_DIR
    / "seasonality_behavior_plan.csv"
)

seasonality.to_csv(
    FINAL_FILE,
    index=False
)

print("\nFinal plan saved to:")
print(FINAL_FILE)

# Final validation
if (
    len(seasonality) == 12
    and len(missing_columns) == 0
    and duplicate_months == 0
    and missing_values == 0
    and invalid_multipliers == 0
):
    print("\n02.12 Step 6 validation: PASSED")
else:
    print("\n02.12 Step 6 validation: FAILED")

02.12 STEP 6 — FINAL SEASONALITY PLAN VALIDATION
Rows: 12
Missing required columns: 0
Duplicate months: 0
Missing values: 0
Invalid multipliers: 0

Final Seasonality Plan:
 month month_name  seasonal_multiplier       season_type
     1    January                 0.95      Post-Holiday
     2   February                 0.90     Soft Baseline
     3      March                 1.00            Normal
     4      April                 1.02     Modest Uplift
     5        May                 1.05     Summer Uplift
     6       June                 1.08 Mid-Year Activity
     7       July                 1.03        Moderation
     8     August                 1.08 Festival Build-Up
     9  September                 1.15      Pre-Festival
    10    October                 1.35      Festive Peak
    11   November                 1.45      Festive Peak
    12   December                 1.60     Year-End Peak

Final plan saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\syntheti